# ERDES: prepare the first baseline inputs

**Goal:** turn the eight verified training clips into consistent model inputs and save
the first pilot's proposed configuration. No model is trained or evaluated here.
This notebook runs independently on CPU; use a free Colab GPU for the later training run.

Working task: **non-RD = 0, RD = 1**. Preserve official partitions. The inspection
sample is deliberately balanced and must not be treated as the pilot training dataset.
Paper v2 documents possible same-patient overlap between partitions; evaluation will
be clip-level and cannot establish generalization to unseen patients.
[Paper, Data Record](https://arxiv.org/html/2508.04735v2#S3).


## 1. Check runtime, storage, and the proposed configuration

Existing Colab libraries are sufficient: numpy, pandas, OpenCV, and matplotlib.
The notebook embeds the same configuration as `configs/baseline_001.json` so it can
run without uploading the project. It saves the configuration with its outputs too.
This is the first engineering pilot, not a final research design or paper reproduction.


In [ ]:
import hashlib
import io
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from urllib.request import urlopen
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

cwd = Path.cwd()
project_root = next((p for p in (cwd, *cwd.parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()), None)
if project_root is None:
    if not Path("/content").is_dir():
        raise RuntimeError("Open from the project folder or select a Colab kernel")
    project_root = Path("/content")
data_dir = project_root / "data" / "erdes"
output_dir = project_root / "outputs" / "baseline_preparation"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
runtime = {"python":sys.version, "numpy":np.__version__, "pandas":pd.__version__,
    "opencv":cv2.__version__, "matplotlib":matplotlib.__version__}
print(runtime)
print("Inputs:", data_dir, "| Results:", output_dir)

def download_checked(url, path, expected_hash, algorithm="sha256", expected_bytes=None):
    if path.exists():
        content = path.read_bytes()
    else:
        with urlopen(url, timeout=90) as response:
            content = response.read()
    assert hashlib.new(algorithm, content).hexdigest() == expected_hash, f"Checksum mismatch: {path.name}"
    if expected_bytes is not None:
        assert len(content) == expected_bytes, f"Unexpected size: {path.name}"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    return content


In [ ]:
config = {
    "experiment_id": "baseline_001_pilot",
    "status": "proposed_not_trained",
    "task": "non_rd_vs_rd",
    "label_mapping": {
        "non_rd": 0,
        "rd": 1
    },
    "seed": 42,
    "split_policy": "preserve official train/val/test membership; no test use during development",
    "preprocessing": {
        "num_frames": 16,
        "height": 128,
        "width": 128,
        "frame_sampling": "uniform frame indices using rint(linspace(0, N-1, T)); repeats for short clips",
        "grayscale": "arithmetic channel mean",
        "padding": "symmetric zero padding to square",
        "resize": "OpenCV INTER_AREA",
        "normalization": "divide by 255",
        "layout": "B,C,T,H,W",
        "dtype": "float32",
        "augmentation": "none"
    },
    "model": {
        "name": "compact_3d_cnn_from_scratch",
        "channels": [
            8,
            16,
            32
        ],
        "head": "global average pooling and one logit",
        "pretrained": False
    },
    "training": {
        "device": "Colab CUDA GPU",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "batch_size": 8,
        "epochs": 5,
        "loss": "unweighted BCEWithLogitsLoss",
        "pilot_training_target": 256,
        "validation": "all 431 supplied validation clips",
        "pilot_selection": "deterministic proportional class sample within official training only; record manifest before fitting"
    },
    "evaluation": {
        "positive_class": "rd",
        "threshold": 0.5,
        "checkpoint_selection": "highest validation average precision; earliest epoch on ties",
        "metrics": [
            "TP",
            "TN",
            "FP",
            "FN",
            "sensitivity",
            "specificity",
            "precision",
            "F1",
            "balanced_accuracy",
            "average_precision",
            "ROC_AUC",
            "accuracy"
        ],
        "final_test": "reserved; no test loading, inference, tuning or checkpoint selection"
    },
    "limitations": [
        "clip-level splits allow same-patient overlap; no unseen-patient generalization claim",
        "eight inspection clips are not a training dataset or a metric estimate",
        "pilot is not a reproduction of the paper or a final experiment"
    ]
}
print(json.dumps(config, indent=2))


## 2. Reuse the same eight verified training clips

Verify the metadata and training CSV, reproduce the fixed inspection selection, and
download missing clips from the pinned source. Existing videos are hash-checked and reused.
No validation or test CSV is read in this preparation notebook.


In [ ]:
release_url = "https://zenodo.org/records/18644370"
metadata_bytes = download_checked(
    f"{release_url}/files/erdes_metadata.csv?download=1", data_dir / "erdes_metadata.csv",
    "95828fb5bdffdbf8173b5bcfc99e8148", algorithm="md5",
)
archive_bytes = download_checked(
    f"{release_url}/files/splits.zip?download=1", data_dir / "splits.zip",
    "c5630e3f056ce358dcd384aaed946e64", algorithm="md5",
)
metadata = pd.read_csv(io.BytesIO(metadata_bytes))
assert len(metadata) == 5381 and metadata["file_path"].is_unique and metadata["clip_id"].is_unique
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    train_bytes = archive.read("splits/non_rd_vs_rd/train.csv")
train = pd.read_csv(io.BytesIO(train_bytes))
assert len(train) == 3873 and train["path"].is_unique
training = train.merge(metadata, left_on="path", right_on="file_path", how="left", validate="one_to_one")
assert training["clip_id"].notna().all(), "Training paths missing from metadata"
assert training["label"].map({0: "non_rd", 1: "rd"}).eq(training["diagnostic_class"]).all(), "Label mismatch"

inspection_seed = 42
subtype_order = ["normal", "pvd", "macula_intact", "macula_detached"]
chosen = []
for subtype in subtype_order:
    candidates = training.loc[training["subtype"].eq(subtype)].copy()
    candidates["selection_rank"] = candidates["clip_id"].map(
        lambda clip_id: hashlib.sha256(f"{inspection_seed}:{clip_id}".encode()).hexdigest()
    )
    assert len(candidates) >= 2
    chosen.append(candidates.sort_values("selection_rank").head(2))
sample = pd.concat(chosen, ignore_index=True).drop(columns="selection_rank")
assert len(sample) == 8 and sample["clip_id"].is_unique
assert set(sample["file_path"]).issubset(set(train["path"]))
display(sample[["clip_id", "diagnostic_class", "subtype", "frame_count", "fps", "width", "height", "duration_seconds"]])
print("PASS: eight training clips selected; no validation/test CSV was read")


In [ ]:
hf_revision = "7867e2504ed3845a46ff7790a2f8c3ffcdc08947"
hf_dataset = "https://huggingface.co/datasets/pcvlab/erdes"

clip_sources = {
    "164267_03674": {
        "sha256": "1986fa40aa2431598ffc858ef2442ae5fb16afa5a37dae047f71cc0d7687795c",
        "bytes": 264720
    },
    "164267_02201": {
        "sha256": "e6c627c62c701dd1b239e58cce71c6463ddc3fc7719f91487b870d5457ba495a",
        "bytes": 559116
    },
    "825315_00216": {
        "sha256": "0409107c7beed9035cf18a8ab81652c41fba462bacde6d7690b478d6f210712a",
        "bytes": 157320
    },
    "825315_00468": {
        "sha256": "29f290ad101cdb7e63388197c332e138bb300cb78049bad1b91bc877e8033737",
        "bytes": 641622
    },
    "102769_00077": {
        "sha256": "17c9e9676950096e55a318307be47d163d8c709d72f26fa8a2612129b8ce76d7",
        "bytes": 607206
    },
    "258033_00019": {
        "sha256": "c84306e773c40f3d6b1b5db8f64a03522c6fdfb9e28d125113dd6bccc3a631a6",
        "bytes": 941360
    },
    "755384_00079": {
        "sha256": "0b2ab744d20bef3b93a1fe99b1ae775fe3f19490d71a12904687ae9144007f7f",
        "bytes": 232445
    },
    "405744_00139": {
        "sha256": "ca3e8ae46e5bb51e0c698a3a93fcbd2617a6e31b4f5169731dad6aeea8088d6b",
        "bytes": 843198
    }
}

assert set(sample["clip_id"]) == set(clip_sources), "Selection differs from the recorded source manifest"
print(f"Selected download size: {sum(v['bytes'] for v in clip_sources.values()) / 1_000_000:.2f} MB")
provenance = []
for row in sample.itertuples(index=False):
    relative_path = PurePosixPath(row.file_path)
    assert not relative_path.is_absolute() and ".." not in relative_path.parts
    source = clip_sources[row.clip_id]
    url = f"{hf_dataset}/resolve/{hf_revision}/clips/{quote(row.file_path, safe='/')}"
    local_path = data_dir / "clips" / row.file_path
    download_checked(url, local_path, source["sha256"], expected_bytes=source["bytes"])
    provenance.append({
        "clip_id": row.clip_id, "file_path": row.file_path, "partition": "train",
        "dataset_label": int(row.label), "subtype": row.subtype, "url": url,
        "sha256": source["sha256"], "bytes": source["bytes"],
    })
    print(row.clip_id, "SHA256 PASS")


## 3. Define the input transformation

1. Select 16 uniformly spaced frame indices across each clip, retaining temporal order.
   Short clips repeat frames; no new motion is synthesized. This samples frame positions,
   not exact timestamps, so variable FPS remains a limitation.
2. Average the three image channels to grayscale, pad symmetrically with zeros to a square,
   and resize to 128×128 using area interpolation. Retain the released crop; do not crop again.
3. Divide pixels by 255, yielding float32 values in [0,1]. No statistics are fitted.

One clip has shape **[1,16,128,128] = [channels,time,height,width]**. A batch adds a
leading dimension. Channel averaging, padding, and division by 255 follow attributed
preprocessing ideas in [paper Table 6](https://arxiv.org/html/2508.04735v2#S4);
our uniform frame selection differs from the paper's temporal interpolation.
These are initial engineering choices to review before full experiments.


In [ ]:
def uniform_frame_indices(frame_count, num_frames):
    if frame_count <= 0 or num_frames <= 0:
        raise ValueError("frame_count and num_frames must be positive")
    return np.rint(np.linspace(0, frame_count - 1, num_frames)).astype(np.int64)

def prepare_frame(frame, image_size):
    if image_size <= 0:
        raise ValueError("image_size must be positive")
    if frame.ndim != 3 or frame.shape[2] != 3:
        raise ValueError("Expected an OpenCV H,W,3 frame")
    gray = frame.mean(axis=2, dtype=np.float32)
    height, width = gray.shape
    side = max(height, width)
    top = (side - height) // 2
    left = (side - width) // 2
    padded = np.pad(gray, ((top, side - height - top), (left, side - width - left)))
    resized = cv2.resize(padded, (image_size, image_size), interpolation=cv2.INTER_AREA)
    return (resized / np.float32(255)).astype(np.float32)

def prepare_clip(path, expected_frames, num_frames=16, image_size=128):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    indices = uniform_frame_indices(expected_frames, num_frames)
    targets = set(indices.tolist())
    selected = {}
    capture = cv2.VideoCapture(str(path))
    count = 0
    try:
        if not capture.isOpened():
            raise ValueError(f"Cannot open video: {path.name}")
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if count in targets:
                selected[count] = prepare_frame(frame, image_size)
            count += 1
    finally:
        capture.release()
    if count != expected_frames:
        raise ValueError(f"{path.name}: decoded {count} frames; metadata expects {expected_frames}")
    clip = np.stack([selected[int(i)] for i in indices], axis=0)[None, ...]
    if not np.isfinite(clip).all() or clip.min() < 0 or clip.max() > 1:
        raise ValueError("Preprocessed values must be finite and in [0,1]")
    return clip, indices


In [ ]:
prepared = []
preparation_rows = []
params = config["preprocessing"]
for row in sample.itertuples(index=False):
    clip, indices = prepare_clip(data_dir / "clips" / row.file_path,
        int(row.frame_count), params["num_frames"], params["height"])
    prepared.append(clip)
    preparation_rows.append({"clip_id":row.clip_id, "subtype":row.subtype,
        "dataset_label":int(row.label), "original_frames":int(row.frame_count),
        "selected_indices":indices.tolist(), "unique_selected_frames":len(set(indices.tolist())),
        "shape":list(clip.shape), "min":float(clip.min()), "max":float(clip.max())})
batch = np.stack(prepared)
labels = sample["label"].to_numpy(dtype=np.float32)
assert batch.shape == (8,1,16,128,128) and batch.dtype == np.float32
assert labels.shape == (8,) and set(labels.tolist()) == {0.0,1.0}
assert np.isfinite(batch).all() and 0 <= batch.min() <= batch.max() <= 1
print("PASS: batch shape", batch.shape, "dtype", batch.dtype, "range", (float(batch.min()),float(batch.max())))
print("Labels:", labels, "| RAM:", round(batch.nbytes / 1_000_000,2), "MB")
print("Short-clip example: 3 source frames -> 16 positions:", uniform_frame_indices(3,16).tolist())
display(pd.DataFrame(preparation_rows).drop(columns=["selected_indices"]))


## 4. Check the transformed inputs

Inspect the first, middle, and last sampled positions after preprocessing. Look for
lost coverage, severe distortion, and artifacts. Labels remain the supplied dataset labels.
Save the image outside Git; clear notebook image outputs before committing.


In [ ]:
fig, axes = plt.subplots(8,3,figsize=(9,20))
for row_number, row in enumerate(sample.itertuples(index=False)):
    for col, t in enumerate([0,8,15]):
        ax = axes[row_number,col]
        ax.imshow(batch[row_number,0,t], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{row.subtype} | original frame {preparation_rows[row_number]['selected_indices'][t]}",fontsize=8)
        ax.axis("off")
fig.tight_layout()
fig.savefig(output_dir / "preprocessed_examples.png",dpi=100)
plt.show()
plt.close(fig)


## 5. Save the preparation evidence

Save the proposed pilot configuration, source hashes, selected clips, preprocessing checks,
and runtime versions. This records inputs and decisions; it is not an experiment result.
Preserve the JSON files and preview image outside temporary Colab storage.


In [ ]:
config_path = output_dir / "baseline_001.json"
config_path.write_text(json.dumps(config,indent=2)+"\n",encoding="utf-8")
report = {"recorded_at_utc":datetime.now(timezone.utc).isoformat(), "stage":"preprocessing_only",
    "runtime":runtime, "config_sha256":hashlib.sha256(config_path.read_bytes()).hexdigest(),
    "metadata_sha256":hashlib.sha256(metadata_bytes).hexdigest(),
    "training_csv_sha256":hashlib.sha256(train_bytes).hexdigest(),
    "video_revision":hf_revision, "clips":provenance, "checks":preparation_rows,
    "batch_shape":list(batch.shape), "batch_dtype":str(batch.dtype),
    "model_trained":False, "validation_evaluated":False, "test_used":False}
report_path = output_dir / "preparation_summary.json"
report_path.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
print("Saved:",config_path)
print("Saved:",report_path)
print("Saved:",output_dir / "preprocessed_examples.png")


## Next: the first GPU pilot

Review [the baseline plan](../docs/BASELINE_PLAN.md). The proposed first model is a compact
3D CNN trained from scratch. First check one forward/backward pass and whether it can
overfit a tiny balanced training batch; then run the recorded pilot on a deterministic
subset of official training data and all supplied validation clips.

The pilot training subset is selected and recorded separately; these eight inspection
clips are only a preprocessing check. Select checkpoints using validation average precision,
report RD sensitivity and specificity plus the other configured metrics, and retain the
0.5 threshold initially. Do not open the test CSV or evaluate test clips during development.
Training dependencies and model code are the next implementation step.
